# k-means from scratch
The class lives in `kmeans.py` next to this Notebook. We test it on generated blobs and on the students.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.datasets import make_blobs
from kmeans import KMeans

def plot(X, labels, km):
    """One colour per cluster, black crosses for the centroids."""
    fig = go.Figure()
    for k in range(km.n_clusters):
        fig.add_trace(go.Scatter(x=X[labels == k, 0], y=X[labels == k, 1], mode="markers", name=f"cluster {k}"))
    fig.add_trace(go.Scatter(x=km.centroids[:, 0], y=km.centroids[:, 1], mode="markers", name="centroid",
                             marker=dict(color="black", size=12, symbol="x")))
    return fig.update_layout(template="simple_white")

## The Euclidean distance as a dot product

In [ ]:
a, b = np.array([1, 2]), np.array([4, 5])
print(np.sqrt(np.dot(b - a, b - a)))           # sqrt(9 + 9) = 4.24
a, b = np.array([1, 2, 3]), np.array([4, 5, 6])
print(np.sqrt(np.dot(b - a, b - a)))           # sqrt(9 + 9 + 9) = 5.196: same code, 3 columns

## The mean of each cluster

In [ ]:
X_small = np.array([[1, 2], [2, 3], [3, 4], [4, 5], [5, 6]])
cluster_group = np.array([0, 1, 0, 0, 1])
print(X_small[cluster_group == 0])               # the rows of cluster 0
print(X_small[cluster_group == 0].mean(axis=0))  # column means: the new centroid of cluster 0
print(X_small[cluster_group == 0].mean())        # without axis: one mean of everything (wrong here)

## Two, three and four blobs

In [ ]:
centres = [(-5, -5), (5, 5), (-2.5, 2.5), (2.5, -2.5)]
for n in (2, 3, 4):
    X, _ = make_blobs(n_samples=100, centers=centres[:n], cluster_std=1, random_state=2)
    km = KMeans(n_clusters=n, max_iter=100, random_state=3)
    labels = km.fit_predict(X)
    print(n, "clusters, finished after", km.n_iter_, "rounds")
plot(X, labels, km)

## Does max_iter matter? Count the rounds

In [ ]:
X, y_true = make_blobs(n_samples=100, centers=centres, cluster_std=1, random_state=2)
rounds = []
for s in range(30):
    km = KMeans(n_clusters=4, max_iter=500, random_state=s)
    km.fit_predict(X)
    rounds.append(km.n_iter_)
print("rounds needed, 30 random starts:", rounds)
print("most rounds ever needed:", max(rounds))

## The students: a bad start and a good start

In [ ]:
students = pd.read_csv("data/student_clustering.csv").values
for seed in (0, 3):
    for max_iter in (100, 500):
        km = KMeans(n_clusters=4, max_iter=max_iter, random_state=seed)
        labels = km.fit_predict(students)
        print(f"seed {seed}, max_iter {max_iter}: {km.n_iter_} rounds, WCSS {km.inertia_:.1f}, sizes {np.bincount(labels)}")

In [ ]:
# The bad start (seed 0): more rounds do not help, it is stuck
km = KMeans(n_clusters=4, max_iter=500, random_state=0)
plot(students, km.fit_predict(students), km)

## The fix: restart several times and keep the lowest WCSS

In [ ]:
best, best_labels = None, None
for s in range(10):
    m = KMeans(n_clusters=4, random_state=s)
    lab = m.fit_predict(students)
    if best is None or m.inertia_ < best.inertia_:
        best, labels = m, lab
print("best of 10 starts: WCSS", round(best.inertia_, 1), "sizes", np.bincount(labels))
plot(students, labels, best)

## Homework: the elbow method with our own class

In [ ]:
wcss = []
for k in range(1, 11):
    # best of 10 random starts for each k
    runs = []
    for s in range(10):
        m = KMeans(n_clusters=k, random_state=s)
        m.fit_predict(students)
        runs.append(m.inertia_)
    wcss.append(min(runs))
print(np.round(wcss, 1))
go.Figure(go.Scatter(x=list(range(1, 11)), y=wcss, mode="lines+markers")).update_layout(
    template="simple_white", xaxis_title="k", yaxis_title="WCSS")